Assignment for Day 3
1. Load stock prices
2. Calculate realized vol: historical, EWMA and GARCH(1,1)
3. 

In [15]:
import pandas as pd
import plotly.express as px
import numpy as np
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint

# load stock prices
ENERGY = ['XOM', 'CVX']    # Exxon, Chevron
TECHNOLOGY = ['MSFT', 'GOOG', 'AMZN']  # Microsoft, Google, Amazon
BANKING = ['JPM', 'GS', 'MS']  # JPMorgan, Goldman Sachs, Morgan Stanley
TICKERS = ENERGY + TECHNOLOGY + BANKING

START_DATE = '2022-01-01'
END_DATE = '2026-03-01'

HISTORY = 42    # nb days to calculate realized vol
ANNUAL = 252   # nb trading days per year


In [2]:

# load stock prices
prices = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)

/opt/miniconda3/envs/algotrading/lib/python3.11/site-packages/yfinance/scrapers/history.py:172: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*********************100%***********************]  8 of 8 completed


In [3]:
close_prices = prices['Close']
close_prices.head()

Ticker,AMZN,CVX,GOOG,GS,JPM,MS,MSFT,XOM
Date,,,,,,,,
2022-01-03,170.404495,100.499298,143.994507,355.345642,144.929504,87.451157,323.160767,54.760242
2022-01-04,167.522003,102.327942,143.341415,366.266754,150.423752,91.003647,317.619507,56.819992
2022-01-05,164.356995,102.993675,136.628784,358.311920,147.673691,88.751701,305.426788,57.526691
2022-01-06,163.253998,103.870071,136.527023,356.783875,149.242584,90.270470,303.013275,58.879753
2022-01-07,162.554001,105.361641,135.984589,357.305176,150.721298,90.820358,303.167725,59.362370


In [4]:
# calculate realized vol
sum_squares = close_prices.pct_change().rolling(window=HISTORY).apply(lambda x: np.sum(x**2))
realized_vol = 100*(np.sqrt(sum_squares)/np.sqrt(HISTORY))*np.sqrt(ANNUAL)

# plot realized vol
fig = px.line(realized_vol, title='Realized Volatility')
fig.update_yaxes(title_text='Volatility (%)')
fig.update_xaxes(title_text=None)
fig.show()


In [5]:
# load VIX index
vix = yf.download('^VIX', start=START_DATE, end=END_DATE, auto_adjust=True)['Close']

# plot vix
fig = px.line(vix, title='VIX')
fig.update_yaxes(title_text='VIX')
fig.update_xaxes(title_text=None)
fig.show()

/opt/miniconda3/envs/algotrading/lib/python3.11/site-packages/yfinance/scrapers/history.py:172: Pandas4Warning:

Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.

[*********************100%***********************]  1 of 1 completed


In [6]:
spy = yf.download('SPY', start=START_DATE, end=END_DATE, auto_adjust=True)['Close']

/opt/miniconda3/envs/algotrading/lib/python3.11/site-packages/yfinance/scrapers/history.py:172: Pandas4Warning:

Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.

[*********************100%***********************]  1 of 1 completed


In [7]:
# calculate 1M SPY returns, bucketed by VIX levels
spy_returns = spy.pct_change(21).dropna()
spy_returns.head()

Ticker,SPY
Date,
2022-02-02,-0.042620
2022-02-03,-0.064810
2022-02-04,-0.042017
2022-02-07,-0.044194
2022-02-08,-0.032504


In [8]:
# concatenate spy_returns and vix - make sure there is no forward-looking information
spy_returns = pd.concat([spy_returns, vix.shift(21)], axis=1)
spy_returns.columns = ['SPY 1M rtn', 'VIX']
spy_returns.head()
# plot spy_returns
fig = px.scatter(spy_returns, x='VIX', y='SPY 1M rtn', title='SPY Returns vs VIX')
fig.show()

/var/folders/82/hh8lkwfx2s5bvc9lnq2rp_0w0000gn/T/ipykernel_41042/246346084.py:2: Pandas4Warning:

Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.



In [9]:
close_prices

Ticker,AMZN,CVX,GOOG,GS,JPM,MS,MSFT,XOM
Date,,,,,,,,
2022-01-03,170.404495,100.499298,143.994507,355.345642,144.929504,87.451157,323.160767,54.760242
2022-01-04,167.522003,102.327942,143.341415,366.266754,150.423752,91.003647,317.619507,56.819992
2022-01-05,164.356995,102.993675,136.628784,358.311920,147.673691,88.751701,305.426788,57.526691
2022-01-06,163.253998,103.870071,136.527023,356.783875,149.242584,90.270470,303.013275,58.879753
2022-01-07,162.554001,105.361641,135.984589,357.305176,150.721298,90.820358,303.167725,59.362370
...,...,...,...,...,...,...,...,...
2026-02-23,205.270004,184.910004,311.690002,887.638611,297.670013,166.800003,384.470001,150.759995
2026-02-24,208.559998,185.339996,310.920013,897.546448,297.299988,168.789993,389.000000,149.259995
2026-02-25,210.639999,184.220001,313.029999,916.556396,303.299988,173.729996,400.600006,149.059998


In [44]:
# plot regression of weekly returns of JPM vs GS
weekly_returns = close_prices.resample('W').last().pct_change()
fig = px.scatter(weekly_returns, x='JPM', y='GS', title='JPM vs GS Weekly Returns')
fig.update_layout(width=900, height=600)
fig.show()


In [21]:
# for all pairs of stocks, test for cointegration and report results

# create a dataframe to store the results
rows = []
df_coint = pd.DataFrame(columns=['ticker1', 'ticker2', 'ADF Statistic', 'p-value', 'beta', 'spread_ADF Statistic', 'spread_p-value'])

for ticker1 in TICKERS:
    for ticker2 in TICKERS:
        if ticker1 != ticker2:
            # run engle-granger test
            result = coint(close_prices[ticker1], close_prices[ticker2])

            # calculate the beta-adjusted spread between the two stocks by running a linear regression
            model = sm.OLS(close_prices[ticker1], close_prices[ticker2])
            results = model.fit()
            # get regression r-squared
            r_squared = results.rsquared
            # get regression p-value
            p_value = results.pvalues[ticker2]
            beta = results.params[ticker2]
            spread = close_prices[ticker1] - beta * close_prices[ticker2]

            # calculate the ADF test statistic for the spread
            adf_result = adfuller(spread)

            # store the results
            rows.append({'ticker1': ticker1, 'ticker2': ticker2, 
                'EG Statistic': result[0], 
                'p-value': result[1], 
                'OLS beta': beta, 
                'OLS r-squared': r_squared,
                'OLS p-value': p_value,
                'spread_ADF Statistic': adf_result[0], 
                'spread_p-value': adf_result[1]})

# print the results
df_coint = pd.DataFrame(rows)


In [22]:
df_coint

,ticker1,ticker2,EG Statistic,p-value,OLS beta,OLS r-squared,OLS p-value,spread_ADF Statistic,spread_p-value
0,XOM,CVX,-1.974296,0.541911,0.703575,0.989219,0.0,-1.099059,0.715447
1,XOM,MSFT,-1.828549,0.615754,0.266495,0.966722,0.0,-1.600524,0.483330
2,XOM,GOOG,-3.437445,0.038358,0.578553,0.927712,0.0,-1.510324,0.528398
3,XOM,AMZN,-2.347201,0.350251,0.578110,0.950062,0.0,-2.334906,0.160957
4,XOM,JPM,-2.751045,0.181759,0.492115,0.931349,0.0,-1.276118,0.640026
5,XOM,GS,-2.915674,0.131634,0.197367,0.914392,0.0,-1.184963,0.680031
6,XOM,MS,-2.783194,0.171090,0.926587,0.948469,0.0,-1.757858,0.401575
7,CVX,XOM,-3.504566,0.031993,1.405990,0.989219,0.0,-1.117523,0.708034
8,CVX,MSFT,-3.174111,0.074365,0.372923,0.947310,0.0,-1.510240,0.528440
9,CVX,GOOG,-3.989533,0.007432,0.805337,0.899520,0.0,-0.612643,0.868099


In [ ]:
# plot JPM and GS price indexes (ie price divided by p[0])
fig = px.line(close_prices[['JPM', 'GS']].div(close_prices[['JPM', 'GS']].iloc[0]), title='JPM and GS price indexes')
fig.update_yaxes(title_text='Price Index')
fig.update_xaxes(title_text=None)

# make the chart size 900x600
fig.update_layout(width=900, height=600)
fig.show()

In [ ]:
beta = 1.0
df_spread = close_prices[['JPM', 'GS']]
df_spread['spread(log)'] = np.log(df_spread['JPM']) - beta * np.log(df_spread['GS'])

# 3M rolling z-score
df_spread['rolling_mean'] = df_spread['spread(log)'].rolling(window=63).mean()
df_spread['rolling_std'] = df_spread['spread(log)'].rolling(window=63).std()
df_spread['z-score'] = (df_spread['spread(log)'] - df_spread['rolling_mean']) / df_spread['rolling_std']

# plot spread
fig = px.line(df_spread, x=df_spread.index, y=['z-score'], title='JPM - GS')
fig.update_yaxes(title_text='Spread (3M z-score)')
fig.update_xaxes(title_text=None)

# make the chart size 900x400
fig.update_layout(width=900, height=400)
fig.update_layout(showlegend=False)
fig.show()